In [1]:
"""
exp03 Step 02: Walk-Forward Evaluation (PROTOCOL.md §6-§9)

No model is trained: the 15 rules of step 01 are fixed. The walk-forward decides WHICH rule to use with information from
earlier quarters only, and measures how that choice does on later quarters.

Per fold (the same 44 folds and validation quarters as exp02):
    1. VALIDATION: simulate every rule on the validation quarter (start invested, as buy-and-hold).
    2. SELECTION: mean validation excess return over the last 4 validation quarters (pooled), per rule.
       Ties: the tighter ATH threshold, then the rule order.
    3. HONEST REPORTING (every run mode, validation data only):
         - every rule chained over the validation quarters (after the fact: optimistic for the best one);
         - the PRIOR-ONLY path: the rule selected at fold f, re-simulated on fold f + 1's validation quarter with its
           baselines (random buy-back with the same exit, random exit with the same buy-back, buy-and-hold, trend rule);
         - time in cash, the minimum detectable effect, the success criteria and the monthly bootstrap on that path.
    4. TEST (run modes "latest" / "history" only): the selected rule on the test window, ONCE, with the same baselines.

STOP (PROTOCOL.md §9) if the prior-only path fails to beat buy-and-hold, or fails to beat its random baselines.

RUN_MODE: "validation_only" (use until the design is frozen), "latest", "history".
Prerequisite: the raw data only. Outputs: store04_experiments/exp03_ath_exit/step02_walk_forward_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp03_ath_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp03_ath_exit.rules import get_rule_candidate_list, simulate_rule_dict, get_rule_metric_dict, get_event_forward_pdf
from experiments.exp03_ath_exit.walk_forward import get_fold_pdf, run_validation_candidate_pdf, get_pooled_selection_pdf, run_window_with_baseline_dict
# IMPORT THE EVALUATION FUNCTIONS
from so.core.evaluation import get_prior_only_pdf, get_candidate_summary_pdf, get_window_return_summary_dict, get_baseline_summary_pdf, \
                               get_chained_daily_return_pdf, get_excess_return_bootstrap_dict, get_success_criteria_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE ("validation_only", "latest" or "history"): NEVER "latest" / "history" BEFORE THE DESIGN IS FROZEN
RUN_MODE_STR = "validation_only"
# DEFINE WHETHER THE TEST WINDOWS ARE EVALUATED
RUN_TEST_BOOL = RUN_MODE_STR in ["latest", "history"]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tTest Windows Evaluated:\t{RUN_TEST_BOOL}")
print(f"ATH thresholds:\t{exp_config.ATH_WITHIN_LIST}\tDelays:\t{exp_config.BUYBACK_DELAY_LIST}\tDips:\t{exp_config.BUYBACK_DIP_LIST}\tMax cash sessions:\t{exp_config.MAX_CASH_SESSIONS}")

Run Mode:	validation_only	Test Windows Evaluated:	False
ATH thresholds:	[0.005, 0.01, 0.02]	Delays:	[5, 20, 60]	Dips:	[0.02, 0.05]	Max cash sessions:	60


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_walk_forward_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp03_ath_exit	Protocol:	1.0	Config Hash:	b8523207d1
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step02_walk_forward_data/
ℹ️ No earlier 'validation' entry of exp03_ath_exit in the trial log (config hash b8523207d1).


In [5]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO BUILD THE DAILY FEATURES AND TARGET (SECONDS; pipeline/step06 SAVES THE SAME TABLE AS A RECORD)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Sessions:\t{len(daily_pdf):,}\tLabelled:\t{int(daily_pdf['y_fwd_positive'].notna().sum()):,}")

min timestamp:	2005-01-03 09:30:00-05:00
max timestamp:	2026-08-13 15:59:00-04:00
Sessions:	5,436	Labelled:	5,416


In [6]:
# CALL FUNCTION TO GET THE FOLDS
fold_pdf = get_fold_pdf(ohlcv_array_dict["session_date_list"])
# SELECT THE FOLDS WHOSE VALIDATION IS NEEDED ("latest" NEEDS THE LAST 4 FOLDS FOR THE POOLED SELECTION)
validation_fold_pdf = fold_pdf.iloc[-exp_config.SELECTION_POOLED_QUARTER_COUNT:] if RUN_MODE_STR == "latest" else fold_pdf
# SELECT THE FOLDS WHOSE TEST WINDOW IS EVALUATED
test_fold_pdf = fold_pdf.iloc[[-1]] if RUN_MODE_STR == "latest" else (fold_pdf if RUN_MODE_STR == "history" else fold_pdf.iloc[0:0])
# SAVE THE SCHEDULE
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Fold Count:\t{len(fold_pdf)}\tValidation Folds:\t{len(validation_fold_pdf)}\tTest Folds:\t{len(test_fold_pdf)}")
# PREVIEW DATAFRAME
fold_pdf[["fold_id", "valid_start", "valid_end", "test_start", "test_end"]]

Fold Count:	44	Validation Folds:	44	Test Folds:	0


,fold_id,valid_start,valid_end,test_start,test_end
0,0,2015-04-17,2015-07-16,2015-08-14,2015-11-13
1,1,2015-07-17,2015-10-16,2015-11-16,2016-02-12
2,2,2015-10-15,2016-01-14,2016-02-16,2016-05-13
3,3,2016-01-19,2016-04-15,2016-05-16,2016-08-12
4,4,2016-04-18,2016-07-15,2016-08-15,2016-11-11
5,5,2016-07-15,2016-10-14,2016-11-14,2017-02-13
6,6,2016-10-14,2017-01-13,2017-02-14,2017-05-12
7,7,2017-01-17,2017-04-13,2017-05-15,2017-08-11
8,8,2017-04-17,2017-07-14,2017-08-14,2017-11-13
9,9,2017-07-17,2017-10-16,2017-11-14,2018-02-13


In [7]:
# LIST TO HOLD THE CANDIDATE TABLES
candidate_pdf_list = []
# ITERATE OVER THE VALIDATION FOLDS
for _, fold_row in validation_fold_pdf.iterrows():
    # DEFINE THE FOLD ID AND START TIME
    fold_id, start_time = int(fold_row["fold_id"]), time.time()
    # CALL FUNCTION TO RUN THE 15 RULES ON THE VALIDATION QUARTER
    fold_candidate_pdf = run_validation_candidate_pdf(fold_row, daily_pdf, ohlcv_array_dict)
    candidate_pdf_list.append(fold_candidate_pdf)
    # LOG EVERY RULE AS A TRIAL (ONE WRITE PER FOLD)
    log_trial_pdf(exp_config, "validation", fold_candidate_pdf, ["ath_within", "rule"], [col for col in fold_candidate_pdf.columns if col.startswith("valid_")], False, fold_pdf)
    # SAVE THE CANDIDATES
    write_csv_file_to_path(fold_candidate_pdf, f"{output_path_str}fold{fold_id:03d}_validation_candidate_data.csv", "W", alert_in=False)
    # DISPLAY INFORMATION
    print(f"Fold {fold_id}:\tvalidation {fold_row['valid_start']} -> {fold_row['valid_end']}\tbuy & hold {fold_candidate_pdf['valid_buy_hold_total_return'].iloc[0]:.2%}"
          f"\trules with an exit {int((fold_candidate_pdf['valid_signal_exit_count'] > 0).sum())}/15\t({(time.time() - start_time):.0f} s)")
# CONCATENATE THE CANDIDATES AND SAVE THEM
candidate_pdf = pd.concat(candidate_pdf_list, ignore_index=True)
write_csv_file_to_path(candidate_pdf, f"{output_path_str}validation_candidate_data.csv", "W")

Fold 0:	validation 2015-04-17 -> 2015-07-16	buy & hold 1.83%	rules with an exit 15/15	(0 s)
Fold 1:	validation 2015-07-17 -> 2015-10-16	buy & hold -4.21%	rules with an exit 10/15	(0 s)
Fold 2:	validation 2015-10-15 -> 2016-01-14	buy & hold -4.13%	rules with an exit 5/15	(1 s)
Fold 3:	validation 2016-01-19 -> 2016-04-15	buy & hold 9.89%	rules with an exit 0/15	(1 s)
Fold 4:	validation 2016-04-18 -> 2016-07-15	buy & hold 3.83%	rules with an exit 15/15	(0 s)
Fold 5:	validation 2016-07-15 -> 2016-10-14	buy & hold -1.49%	rules with an exit 15/15	(0 s)
Fold 6:	validation 2016-10-14 -> 2017-01-13	buy & hold 5.87%	rules with an exit 15/15	(0 s)
Fold 7:	validation 2017-01-17 -> 2017-04-13	buy & hold 2.80%	rules with an exit 15/15	(0 s)
Fold 8:	validation 2017-04-17 -> 2017-07-14	buy & hold 5.26%	rules with an exit 15/15	(0 s)
Fold 9:	validation 2017-07-17 -> 2017-10-16	buy & hold 3.92%	rules with an exit 15/15	(0 s)
Fold 10:	validation 2017-10-17 -> 2018-01-16	buy & hold 8.47%	rules with an exi

In [8]:
"""
After-the-fact view (OPTIMISTIC for the best rule: it is the best of 15 chosen with hindsight).
"""
# CALL FUNCTION TO SUMMARIZE EVERY RULE OVER THE VALIDATION QUARTERS
candidate_summary_pdf = get_candidate_summary_pdf(candidate_pdf, ["ath_within", "rule"], ["valid_signal_exit_count", "valid_cash_session_pct", "valid_positive_episode_pct"])
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# DISPLAY THE CHAINED EXCESS RETURN GRID (THRESHOLD x BUY-BACK)
display(candidate_summary_pdf.pivot(index="ath_within", columns="rule", values="chained_excess_return").round(4))
# PREVIEW DATAFRAME
candidate_summary_pdf.round(4)

rule,delay_20,delay_5,delay_60,dip_2pct,dip_5pct
ath_within,,,,,
0.005,-0.2652,-0.6627,-1.2873,-0.4945,-1.1807
0.010,-0.9078,-0.5346,-1.3625,-1.0099,-1.2556
0.020,-0.5946,-0.9650,-1.4444,-1.0195,-1.3449


,ath_within,rule,window_count,chained_return,chained_buy_hold_return,chained_excess_return,windows_beating_buy_hold,sign_test_p_value,mean_excess_return,std_excess_return,t_stat,mde_per_window,mde_per_year,mean_valid_signal_exit_count,mean_valid_cash_session_pct,mean_valid_positive_episode_pct
0,0.005,delay_20,44,2.1686,2.4338,-0.2652,12,0.9994,-0.0026,0.0508,-0.3388,0.0214,0.0858,1.3864,0.3884,0.3851
1,0.005,dip_2pct,44,1.9392,2.4338,-0.4945,13,0.9982,-0.0041,0.0336,-0.8198,0.0142,0.0567,0.9318,0.4168,0.4253
2,0.010,delay_5,44,1.8992,2.4338,-0.5346,15,0.9887,-0.0043,0.0278,-1.0364,0.0117,0.0470,4.3409,0.3365,0.4654
3,0.020,delay_20,44,1.8391,2.4338,-0.5946,11,0.9998,-0.0055,0.0577,-0.6269,0.0244,0.0975,2.0000,0.5692,0.3308
4,0.005,delay_5,44,1.7711,2.4338,-0.6627,12,0.9994,-0.0053,0.0209,-1.6668,0.0088,0.0353,3.2273,0.2534,0.4514
5,0.010,delay_20,44,1.5260,2.4338,-0.9078,10,0.9999,-0.0079,0.0551,-0.9491,0.0232,0.0930,1.6136,0.4700,0.3441
6,0.020,delay_5,44,1.4687,2.4338,-0.9650,10,0.9999,-0.0081,0.0364,-1.4823,0.0153,0.0614,5.5909,0.4310,0.4022
7,0.010,dip_2pct,44,1.4238,2.4338,-1.0099,13,0.9982,-0.0088,0.0350,-1.6635,0.0148,0.0591,0.9773,0.4727,0.4247
8,0.020,dip_2pct,44,1.4143,2.4338,-1.0195,12,0.9994,-0.0090,0.0421,-1.4120,0.0178,0.0710,1.2500,0.5363,0.3990
9,0.005,dip_5pct,44,1.2531,2.4338,-1.1807,8,1.0000,-0.0105,0.0373,-1.8695,0.0158,0.0631,0.7273,0.4760,0.2586


In [9]:
"""
Prior-only path (the honest validation estimate): the rule selected at fold f (pooled over the 4 quarters up to f) is
evaluated on fold f + 1's validation quarter, with its baselines. No test window is read.
"""
# CALL FUNCTION TO SELECT ONE RULE PER FOLD (POOLED OVER THE LAST 4 VALIDATION QUARTERS)
selection_pdf = get_pooled_selection_pdf(candidate_pdf)
write_csv_file_to_path(selection_pdf, f"{output_path_str}selection_data.csv", "W", alert_in=False)
# CALL FUNCTION TO GET THE PRIOR-ONLY PATH (SELECTION OF FOLD f EVALUATED ON FOLD f + 1)
prior_only_pdf = get_prior_only_pdf(candidate_pdf, selection_pdf, ["ath_within", "rule"])
# LISTS TO HOLD THE BASELINES AND THE DAILY EQUITY OF THE PATH
prior_baseline_pdf_list, prior_equity_pdf_list, prior_buy_hold_equity_pdf_list = [], [], []
# ITERATE OVER THE EVALUATED FOLDS
for _, prior_row in prior_only_pdf.iterrows():
    # CALL FUNCTION TO RE-SIMULATE THE SELECTED RULE ON THE QUARTER WITH ITS BASELINES
    fold_row = fold_pdf[fold_pdf["fold_id"] == prior_row["fold_id"]].iloc[0]
    window_dict = run_window_with_baseline_dict(fold_row, prior_row, daily_pdf, ohlcv_array_dict, "valid", alert_in=False)
    # CHECK THE RE-SIMULATION EQUALS THE LOGGED CANDIDATE
    assert np.isclose(window_dict["metric_dict"]["total_return"], prior_row["valid_total_return"]), f"fold {prior_row['fold_id']}: re-simulation differs"
    # COLLECT THE RESULTS
    prior_baseline_pdf_list.append(window_dict["baseline_pdf"].assign(fold_id=int(prior_row["fold_id"])))
    prior_equity_pdf_list.append(window_dict["daily_equity_pdf"])
    prior_buy_hold_equity_pdf_list.append(window_dict["buy_hold_daily_equity_pdf"])
# CONCATENATE THE BASELINES AND SAVE THE PATH
prior_baseline_pdf = pd.concat(prior_baseline_pdf_list, ignore_index=True)
write_csv_file_to_path(prior_only_pdf, f"{output_path_str}prior_only_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_baseline_pdf, f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
# SUMMARIZE THE PATH (QUARTERS), THE BASELINES (INFORMATION TEST) AND THE DAILY CRITERIA
prior_summary_dict = get_window_return_summary_dict(prior_only_pdf["valid_total_return"].to_numpy(), prior_only_pdf["valid_buy_hold_total_return"].to_numpy())
baseline_summary_pdf = get_baseline_summary_pdf(prior_baseline_pdf, random_prefix_str_list_in=("random_buyback_", "random_exit_"))
prior_return_pdf, prior_buy_hold_return_pdf = get_chained_daily_return_pdf(prior_equity_pdf_list), get_chained_daily_return_pdf(prior_buy_hold_equity_pdf_list)
baseline_chained_dict = {row["baseline"]: row["chained_return"] for _, row in baseline_summary_pdf.iterrows() if row["baseline"].startswith("random")}
prior_success_dict = get_success_criteria_dict(prior_return_pdf, prior_buy_hold_return_pdf, baseline_chained_dict)
prior_bootstrap_dict = get_excess_return_bootstrap_dict(prior_return_pdf, prior_buy_hold_return_pdf)
# DISPLAY INFORMATION
print(f"Prior-only path ({prior_summary_dict['window_count']} quarters):\t{prior_summary_dict['chained_return']:.2%} vs buy & hold {prior_summary_dict['chained_buy_hold_return']:.2%}"
      f"\t(beats buy & hold in {prior_summary_dict['windows_beating_buy_hold']} quarters, one-sided sign test p = {prior_summary_dict['sign_test_p_value']:.3f})")
print(f"Mean excess per quarter:\t{prior_summary_dict['mean_excess_return']:.2%} (SD {prior_summary_dict['std_excess_return']:.2%}, t = {prior_summary_dict['t_stat']:.2f})"
      f"\tMinimum detectable effect:\t{prior_summary_dict['mde_per_window']:.2%} per quarter, about {prior_summary_dict['mde_per_year']:.1%} per year")
print(f"Mean time in cash:\t{prior_only_pdf['valid_cash_session_pct'].mean():.1%}\tQuarters with an exit:\t{int((prior_only_pdf['valid_signal_exit_count'] > 0).sum())}")
print(f"PRIMARY on the prior-only path:\t{prior_success_dict['primary_success']}\tComparable return:\t{prior_success_dict['comparable_return']}"
      f"\tSecondary Sharpe / drawdown:\t{prior_success_dict['secondary_sharpe']} / {prior_success_dict['secondary_drawdown']}")
print(f"Excess return 95% interval (monthly bootstrap):\t[{prior_bootstrap_dict['ci_low']:.2%}, {prior_bootstrap_dict['ci_high']:.2%}]\tstatistically supported: {prior_bootstrap_dict['supported']}")
# PREVIEW THE INFORMATION TEST (THE RULE'S CHAINED RETURN vs EACH BASELINE)
baseline_summary_pdf.round(4)

Prior-only path (43 quarters):	143.05% vs buy & hold 237.21%	(beats buy & hold in 14 quarters, one-sided sign test p = 0.993)
Mean excess per quarter:	-0.85% (SD 3.17%, t = -1.75)	Minimum detectable effect:	1.35% per quarter, about 5.4% per year
Mean time in cash:	38.6%	Quarters with an exit:	31
PRIMARY on the prior-only path:	False	Comparable return:	False	Secondary Sharpe / drawdown:	False / False
Excess return 95% interval (monthly bootstrap):	[-400.79%, 12.44%]	statistically supported: False


,baseline,chained_return,model_beats_random_run_share,random_run_count,model_chained_return,model_beats
0,buy_hold,2.3721,NaN,NaN,1.4305,False
1,trend_ma200,1.0616,NaN,NaN,1.4305,True
2,random_buyback_* (median of runs),1.8048,0.05,20.0,1.4305,False
3,random_exit_* (median of runs),0.5907,0.95,20.0,1.4305,True


In [10]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs BUY AND HOLD (CHAINED QUARTERS)
plot_pdf = prior_baseline_pdf[prior_baseline_pdf["baseline"].isin(["model", "buy_hold", f"trend_ma{exp_config.TREND_RULE_MA_SESSIONS}"])] \
    .sort_values("fold_id").assign(growth=lambda df: df.groupby("baseline")["total_return"].transform(lambda s: np.cumprod(1 + s)))
fig = px.line(plot_pdf, x="fold_id", y="growth", color="baseline", markers=True, template="plotly_dark", width=1100, height=500,
              title="Prior-Only Path On Validation Quarters (growth of 1, chained quarters; model = the selected rule)")
fig.show()
# PREVIEW THE SELECTED RULE PER FOLD
selection_pdf[["fold_id", "ath_within", "rule", "pooled_valid_excess_return", "pooled_quarter_count"]].tail(12)

,fold_id,ath_within,rule,pooled_valid_excess_return,pooled_quarter_count
32,32,0.005,delay_5,0.000000,4
33,33,0.005,delay_5,0.000000,4
34,34,0.010,delay_5,0.004956,4
35,35,0.005,delay_20,-0.007804,4
36,36,0.005,delay_5,-0.019693,4
37,37,0.005,delay_5,-0.015627,4
38,38,0.005,delay_5,-0.012811,4
39,39,0.005,delay_5,0.001194,4
40,40,0.020,dip_5pct,0.011826,4
41,41,0.010,delay_60,0.010463,4


In [11]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE RULES ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} quarters", "path": "prior-only on validation"},
               {**prior_summary_dict, **{f"success_{key}": value for key, value in prior_success_dict.items() if not isinstance(value, dict)},
                "bootstrap_ci_low": prior_bootstrap_dict["ci_low"], "bootstrap_ci_high": prior_bootstrap_dict["ci_high"],
                "baseline_chained_return_dict": baseline_chained_dict, "best_after_the_fact_chained_return": float(candidate_summary_pdf["chained_return"].iloc[0])},
               False, note_str_in="exp03 step02 validation summary (prior-only path, baselines, MDE)")

In [12]:
# LISTS TO HOLD THE TEST RESULTS
fold_summary_dict_list, strategy_equity_pdf_list, buy_hold_equity_pdf_list, baseline_pdf_list = [], [], [], []
# ITERATE OVER THE TEST FOLDS (NONE IN validation_only MODE)
for _, fold_row in test_fold_pdf.iterrows():
    # COLLECT THE FOLD AND ITS SELECTED RULE
    fold_id = int(fold_row["fold_id"])
    selection_row = selection_pdf[selection_pdf["fold_id"] == fold_id].iloc[0]
    print(f"\nFold {fold_id}:\ttest {fold_row['test_start']} -> {fold_row['test_end']}\tselected ATH within {selection_row['ath_within']}, {selection_row['rule']}")
    # CALL FUNCTION TO EVALUATE THE TEST WINDOW ONCE
    test_dict = run_window_with_baseline_dict(fold_row, selection_row, daily_pdf, ohlcv_array_dict, "test")
    # LOG THE TEST EVALUATION
    log_trial_dict(exp_config, "test", {"ath_within": selection_row["ath_within"], "rule": selection_row["rule"]}, test_dict["metric_dict"], True, fold_row)
    # SAVE THE TEST OUTPUTS
    for name_str in ["baseline_pdf", "cost_sensitivity_pdf", "transaction_pdf", "episode_pdf", "daily_equity_pdf"]:
        write_csv_file_to_path(test_dict[name_str], f"{output_path_str}fold{fold_id:03d}_test_{name_str.replace('_pdf', '')}_data.csv", "W", alert_in=False)
    # COLLECT THE RESULTS
    strategy_equity_pdf_list.append(test_dict["daily_equity_pdf"])
    buy_hold_equity_pdf_list.append(test_dict["buy_hold_daily_equity_pdf"])
    baseline_pdf_list.append(test_dict["baseline_pdf"].assign(fold_id=fold_id))
    fold_summary_dict_list.append({"fold_id": fold_id, "test_start": fold_row["test_start"], "test_end": fold_row["test_end"],
                                   "ath_within": selection_row["ath_within"], "rule": selection_row["rule"], **test_dict["metric_dict"]})
# CONVERT THE FOLD SUMMARIES
fold_summary_pdf = pd.DataFrame(fold_summary_dict_list)

In [13]:
# IF THERE ARE TEST RESULTS
if not fold_summary_pdf.empty:
    # SAVE THE FOLD SUMMARY
    write_csv_file_to_path(fold_summary_pdf, f"{output_path_str}fold_summary_data.csv", "W")
    # CHAIN THE DAILY RETURNS OF THE TEST WINDOWS AND SUMMARIZE THE BASELINES
    strategy_return_pdf, buy_hold_return_pdf = get_chained_daily_return_pdf(strategy_equity_pdf_list), get_chained_daily_return_pdf(buy_hold_equity_pdf_list)
    test_baseline_summary_pdf = get_baseline_summary_pdf(pd.concat(baseline_pdf_list, ignore_index=True), random_prefix_str_list_in=("random_buyback_", "random_exit_"))
    test_baseline_chained_dict = {row["baseline"]: row["chained_return"] for _, row in test_baseline_summary_pdf.iterrows() if row["baseline"].startswith("random")}
    # APPLY THE SUCCESS CRITERIA AND THE BOOTSTRAP
    success_dict = get_success_criteria_dict(strategy_return_pdf, buy_hold_return_pdf, test_baseline_chained_dict)
    bootstrap_dict = get_excess_return_bootstrap_dict(strategy_return_pdf, buy_hold_return_pdf)
    # DISPLAY INFORMATION
    print(f"Chained total return:\trule {success_dict['strategy_total_return']:.2%} | buy & hold {success_dict['buy_hold_total_return']:.2%}\tPRIMARY:\t{success_dict['primary_success']}")
    print(f"Excess return 95% interval (monthly bootstrap):\t[{bootstrap_dict['ci_low']:.2%}, {bootstrap_dict['ci_high']:.2%}]\tstatistically supported: {bootstrap_dict['supported']}")
    print(f"Comparable return:\t{success_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{success_dict['secondary_sharpe']} / {success_dict['secondary_drawdown']}")
    print(f"Random baselines beaten:\t{success_dict['baselines_beaten']}")
# PREVIEW DATAFRAME
fold_summary_pdf

""


In [14]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
1,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
2,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
3,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
4,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
5,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
6,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
